# Bit-flip rate vs cat size, plain RWA model, with Krylov-Schur

The `alpha_sq` sweep of `notebooks/sweeps/sweep_alpha.ipynb` (same grid, same per-band Krylov-Schur settings), run on the **plain RWA model** of `test_RWA_0.ipynb` instead of the full ATS, and compared point by point with the **one-mode model** (buffer adiabatically eliminated, scipy `eigs`).

- **RWA model** (time independent): `H = epsilon_d/2 (b + b†) + g (a² b† + a†² b)`, jumps `[sqrt(kappa_1) a, sqrt(kappa_b) b]`, parameters from `floquet_lindblad.models.ats._derived_params`.
- **One-mode model**: `H = 0`, jumps `[sqrt(kappa_2) (a² + epsilon_d/(2g)), sqrt(kappa_1) a]`, `kappa_2 = 4 g²/kappa_b`; `n_a = 60` (converged to ~1e-4 at `alpha_sq = 7`).
- **Two truncations**, each run as **one job on one GPU, points one after the other**:
  - `ats`: the ATS ladder of `sweep_alpha.ipynb` (`n_a` / `n_b` = 20/10, 25/12, 35/16, 38/19);
  - `small`: per point, the smallest `n_a` at which the one-mode rate is converged to `1e-3`, and `n_b = 5` (the RWA buffer stays close to vacuum).

## What differs from the ATS sweep

The RWA rates are much smaller than the ATS ones: the one-mode estimate goes from `1e-6` (`alpha_sq = 1.5`) to `6e-13` (`alpha_sq = 7`). With `T_block = 2π/w_a`, `1 - mu` per block falls to `1.5e-13`, far below any integrator tolerance. The Krylov loop only has to converge the eigenvector, which still works (its residual is relative to `|mu| ~ 1`). The **rate**, though, is read off the final Rayleigh quotient `1 - mu**n`. So `n_blocks_final` is set per point from the one-mode estimate so that `1 - mu**n ~ 1e5 x rtol_final` (the ATS ladder's rule), capped at `N_BLOCKS_MAX`. The plan cell prints the expected resolution `(1 - mu**n) / rtol_final` per point. Where it is only a few hundred, the rate is good to roughly a percent at best.

## Running it

Kernel: the workspace pixi environment (Python 3.13.x). Vault session as for the other cluster notebooks. Run all cells in a fresh kernel. The `small` sweep runs first (it's cheap and gives fast feedback); results are pickled to `results/sweep_alpha_RWA_0.pkl` after each truncation.

In [ ]:
# Cluster bootstrap, same as test_RWA_0.ipynb / notebooks/solvers/krylov_schur.ipynb.
import os
import subprocess
import sys

assert sys.version_info[:2] == (3, 13), (
    f"kernel runs Python {sys.version.split()[0]}; the cluster needs 3.13.x. "
    "Select the workspace pixi environment as the kernel."
)

# Ray reads RAY_AUTH_MODE once, at its first import, and caches it. Setting it
# here works *only* because nothing has imported ray yet in a fresh kernel. If
# you have already run anything that imports ray, restart the kernel first.
os.environ["RAY_AUTH_MODE"] = "token"
os.environ.setdefault("VAULT_ADDR", "https://vault.int.alice-bob.com")

# Mint the cluster token from Vault here, so the notebook also works from a
# kernel that was not launched from a shell with RAY_AUTH_TOKEN exported (which
# is the normal case in VS Code). `vault login` is interactive, so it stays a
# manual, once-a-week step.
if not os.environ.get("RAY_AUTH_TOKEN"):
    try:
        os.environ["RAY_AUTH_TOKEN"] = subprocess.run(
            ["vault", "kv", "get", "-field=token", "kv-it/prod/ray/auth-token"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "Could not mint the cluster token from Vault. In a terminal, run:\n"
            "  export VAULT_ADDR=https://vault.int.alice-bob.com\n"
            "  vault login -method=oidc role=research_core_user\n"
            "then re-run this cell (no kernel restart needed).\n"
            f"{(getattr(exc, 'stderr', '') or '').strip()}"
        ) from exc

import ray

from anb_compute import ray as acr

print("python", sys.version.split()[0], "| ray", ray.__version__)

import pathlib


def _find_repo():
    """Locate the repo checkout without hardcoding an absolute path.

    Notebooks have no __file__, so search upward from the working directory --
    which VS Code sets to either the notebook's folder or the workspace root,
    depending on `jupyter.notebookFileRoot`. Both are handled.
    """
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "src" / "floquet_lindblad").is_dir():
            return base
        for hit in sorted(base.glob("*/src/floquet_lindblad")):
            return hit.parents[1]
    raise RuntimeError(f"no checkout with src/floquet_lindblad found above {here}")


REPO = _find_repo()

# Shipped by PATH, not by name: a bare module name that resolves to a
# non-editable site-packages install ships a frozen copy, and local edits would
# never reach the worker. These prepend to the worker's sys.path, so the
# dynamiqs here shadows the image's stock 0.3.6 (which lacks `mesolve_fast`).
PKG = str(REPO / "src" / "floquet_lindblad")
# The patched dynamiqs checkout, expected beside the repo; $DYNAMIQS_SRC wins.
DYNAMIQS = os.environ.get("DYNAMIQS_SRC") or str(
    REPO.parent / "dynamiqs" / "dynamiqs"
)
for _p in (PKG, DYNAMIQS):
    if not pathlib.Path(_p).is_dir():
        raise RuntimeError(f"not a directory, cannot ship to workers: {_p}")

RUNTIME_ENV = acr.build_runtime_env(py_modules=[PKG, DYNAMIQS])

### Derived parameters

`g`, `kappa_1`, `kappa_2`, `epsilon_d` and `T_drive` come from `ats._derived_params`, which this client cannot import (`floquet_lindblad` pulls in jax and dynamiqs). A short CPU-only job fetches them, so the whole plan is printed before a GPU is asked for, with no constants copied.

In [ ]:
import numpy as np

EPS_P = 0.1
ALPHA_SQ_LIST = [float(a) for a in np.linspace(1.5, 7, 12)]  # 1.5, 2, ..., 7, as the ATS sweep


def derived_params(alpha_sq_list, eps_p):
    """`ats._derived_params` for each point, as plain floats (runs on a worker)."""
    from floquet_lindblad.models.ats import E_J, KAPPA_B, PHI_A, PHI_B, W_A, _derived_params

    out = {}
    for alpha_sq in alpha_sq_list:
        params, _ = _derived_params(
            alpha_sq, W_A, KAPPA_B, E_J, PHI_A, PHI_B, eps_p, n_periods=1
        )
        out[alpha_sq] = {kk: float(v) for kk, v in params.items()}
    return out


PARAMS = acr.run(derived_params, ALPHA_SQ_LIST, EPS_P, runtime_env=RUNTIME_ENV, poll_seconds=5)
p0 = PARAMS[ALPHA_SQ_LIST[0]]
print(f"g = {p0['g']:.6g}, kappa_2 = {p0['kappa_2']:.6g}, kappa_1 = {p0['kappa_1']:.6g}, "
      f"kappa_b = {p0['kappa_b']:.6g}, T_drive = {p0['T_drive']:.6g}")

### One-mode reference

Same code as in `test_RWA_0.ipynb`: the parity-odd block of the one-mode `L`, shift-invert `eigs` at 0. It runs here on the client, well under a second per point. It gives the reference curve, the `small` truncation's `n_a`, and the expected `1 - mu` per block used to plan the final Rayleigh quotient.

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix, diags, eye, kron
from scipy.sparse.linalg import eigs


def liouvillian(H, jump_ops):
    """Column-major vectorized Lindbladian, d vec(rho)/dt = L @ vec(rho) (as `build_liouvillian` above)."""
    n = H.shape[0]
    I = eye(n, format="csr")
    H = csr_matrix(H)
    L = -1j * (kron(I, H) - kron(H.T, I))
    for J in map(csr_matrix, jump_ops):
        JdJ = J.conj().T @ J
        L += kron(J.conj(), J) - 0.5 * (kron(I, JdJ) + kron(JdJ.T, I))
    return L.tocsr()


def one_mode_bit_flip(n_a, g, epsilon_d, kappa_2, kappa_1, k=4):
    """Bit-flip rate of the adiabatically eliminated (one-mode) model, by shift-invert `eigs`.

    Eliminating b from H = g (a^2 + epsilon_d / (2g)) b^dag + h.c. (the RWA model
    above) gives the two-photon jump sqrt(kappa_2) (a^2 + epsilon_d / (2g)): the
    same cat, |alpha|^2 = alpha_sq, as the two-mode model. The parity
    rho -> P rho P is a symmetry, and the bit-flip mode is the slowest mode of the
    odd block. That block does not hold the steady state, so it is not singular
    and sigma = 0 is safe.
    """
    a = diags(np.sqrt(np.arange(1, n_a)), 1, format="csr", dtype=complex)
    jump_ops = [
        np.sqrt(kappa_2) * (a @ a + epsilon_d / (2 * g) * eye(n_a, format="csr")),
        np.sqrt(kappa_1) * a,
    ]
    L = liouvillian(csr_matrix((n_a, n_a), dtype=complex), jump_ops)

    p = (-1.0) ** np.arange(n_a)
    odd = np.flatnonzero(np.kron(p, p) < 0)  # rho[m, n] with n_m + n_n odd
    vals = eigs(L[odd][:, odd].tocsc(), k=k, sigma=0, which="LM", return_eigenvectors=False)
    lam = vals[np.argmax(vals.real)]
    return {"rate_bf": -lam.real, "lam": lam, "vals_odd": vals, "dim": n_a**2}


N_A_REF = 60     # one-mode reference truncation: 50 -> 60 moves the rate by 1e-4 at alpha_sq = 7
SMALL_TOL = 1e-3  # `small` truncation: smallest n_a with the one-mode rate within this of the reference

ONE_MODE_ARGS = {
    a: dict(g=p["g"], epsilon_d=p["epsilon_d"], kappa_2=p["kappa_2"], kappa_1=p["kappa_1"])
    for a, p in PARAMS.items()
}
RATE_1M = {a: one_mode_bit_flip(N_A_REF, **ONE_MODE_ARGS[a])["rate_bf"] for a in ALPHA_SQ_LIST}


def smallest_n_a(alpha_sq, tol=SMALL_TOL, n_min=10):
    """Smallest n_a at which the one-mode rate is within `tol` of the reference."""
    ref = RATE_1M[alpha_sq]
    for n in range(n_min, N_A_REF):
        if abs(one_mode_bit_flip(n, **ONE_MODE_ARGS[alpha_sq])["rate_bf"] / ref - 1) < tol:
            return n
    return N_A_REF


N_A_SMALL = {a: smallest_n_a(a) for a in ALPHA_SQ_LIST}

print(f"{'alpha_sq':>8} {'rate (one mode)':>16} {'n_a for 1e-3':>13}")
for a in ALPHA_SQ_LIST:
    print(f"{a:8.2f} {RATE_1M[a]:16.4e} {N_A_SMALL[a]:13d}")

### The plan

The bands and their Krylov-Schur settings are those of `notebooks/sweeps/sweep_alpha.ipynb` (truncation, `m`/`k`, loop and final tolerances, `res_tol`, cycle budget). Only `n_blocks_final` changes: it is set per point from the one-mode rate (see the top cell), at least the ATS band's value and at most `N_BLOCKS_MAX`.

In [ ]:
# Copied from notebooks/sweeps/sweep_alpha.ipynb (its reasoning is there).
SCHEDULE = [
    dict(alpha_sq_max=2.0, n_a=20, n_b=10, m=40, k=10,
         rtol_loop=1e-7, atol_loop=1e-8, res_tol_loop=1e-6,
         rtol_final=1e-9, atol_final=1e-10, res_tol=1e-8,
         n_blocks_final=10, max_cycles=25),
    dict(alpha_sq_max=3.5, n_a=25, n_b=12, m=40, k=10,
         rtol_loop=1e-8, atol_loop=1e-9, res_tol_loop=1e-7,
         rtol_final=1e-10, atol_final=1e-11, res_tol=1e-9,
         n_blocks_final=20, max_cycles=30),
    dict(alpha_sq_max=6.5, n_a=35, n_b=16, m=50, k=12,
         rtol_loop=1e-9, atol_loop=1e-10, res_tol_loop=1e-8,
         rtol_final=1e-11, atol_final=1e-12, res_tol=1e-10,
         n_blocks_final=50, max_cycles=35),
    dict(alpha_sq_max=np.inf, n_a=38, n_b=19, m=60, k=14,
         rtol_loop=1e-10, atol_loop=1e-11, res_tol_loop=1e-9,
         rtol_final=1e-12, atol_final=1e-13, res_tol=1e-11,
         n_blocks_final=100, max_cycles=40),
]

N_B_SMALL = 5
RQ_FACTOR = 1e5      # aim for 1 - mu**n ~ RQ_FACTOR * rtol_final, as the ATS ladder
N_BLOCKS_MAX = 2000  # ~500 time units of integration for the final Rayleigh quotient

TRUNCATIONS = {
    "small": lambda a, band: (N_A_SMALL[a], N_B_SMALL),
    "ats": lambda a, band: (band["n_a"], band["n_b"]),
}


def point_settings(alpha_sq, truncation):
    """Settings of one point: its ATS band, its truncation, its final-block length."""
    band = next(b for b in SCHEDULE if alpha_sq < b["alpha_sq_max"])
    n_a, n_b = TRUNCATIONS[truncation](alpha_sq, band)
    gap = RATE_1M[alpha_sq] * PARAMS[alpha_sq]["T_drive"]  # expected 1 - mu per block
    n_blocks = int(np.clip(np.ceil(RQ_FACTOR * band["rtol_final"] / gap),
                           band["n_blocks_final"], N_BLOCKS_MAX))
    settings = {kk: v for kk, v in band.items() if kk not in ("alpha_sq_max", "n_a", "n_b")}
    settings.update(alpha_sq=alpha_sq, n_a=n_a, n_b=n_b, n_blocks_final=n_blocks)
    return settings


PLANS = {name: [point_settings(a, name) for a in ALPHA_SQ_LIST] for name in TRUNCATIONS}
COMMON = dict(eps_p=EPS_P, seed=0, verbose=False)  # T_block = 2 pi / w_a (worker default)

for name, points in PLANS.items():
    print(f"=== truncation: {name} ===")
    print(f"{'alpha_sq':>8} {'n_a':>4} {'n_b':>4} {'N':>5} {'m':>4} {'k':>3} {'rtol_final':>11} "
          f"{'blocks':>7} {'RQ time':>8} {'resolution':>11} {'basis GB':>9}")
    for p in points:
        N = p["n_a"] * p["n_b"]
        T_rq = p["n_blocks_final"] * PARAMS[p["alpha_sq"]]["T_drive"]
        resolution = RATE_1M[p["alpha_sq"]] * T_rq / p["rtol_final"]  # (1 - mu**n) / rtol_final
        print(f"{p['alpha_sq']:8.2f} {p['n_a']:4d} {p['n_b']:4d} {N:5d} {p['m']:4d} {p['k']:3d} "
              f"{p['rtol_final']:11.0e} {p['n_blocks_final']:7d} {T_rq:8.1f} {resolution:11.1e} "
              f"{(p['m'] + 1) * N**2 * 16 / 1024**3:9.2f}")
    print()

### One point, and the sequential sweep

`run_krylov_schur_rwa` is the worker of `test_RWA_0.ipynb`, unchanged except for the `verbose` switch. `sweep_sequential` runs as one job holding one GPU and solves the points one after the other. A point that raises is recorded as an error, and the sweep carries on.

In [ ]:
def run_krylov_schur_rwa(
    *,
    alpha_sq,
    eps_p,
    n_a,
    n_b,
    T_block=None,
    # Krylov-Schur (defaults: those of `solve_point_ks`)
    m=50,
    k=12,
    res_tol=1e-10,
    max_cycles=35,
    rtol_loop=1e-9,
    atol_loop=1e-10,
    res_tol_loop=1e-8,
    max_cycles_loop=None,
    rtol_final=1e-11,
    atol_final=1e-12,
    n_blocks_final=50,
    seed=0,
    k_top=3,
    verbose=True,
):
    """`solve_point_ks`, on the plain RWA model instead of the full ATS, on one GPU worker.

    Same two tolerance phases, same final Rayleigh quotient on `n_blocks_final`
    blocks, and the same result keys (`rate_bf`, `res_rel_n`, `tail_a`, ...), so
    the two results can be compared key by key. The model is time independent,
    so there is no frame transformation and no `output_phase`. `T_block` is a
    free time step (default: one ATS drive period, 2 pi / w_a).

    Copied from `test_RWA_0.ipynb`, plus `verbose` (per-cycle printing; the
    lines are kept in `log` either way).

    Extra key: `odd_fraction` = ||rho - P rho P|| / (2 ||rho||) of the target,
    with P the storage-mode parity: 1 for a bit-flip (parity-odd) mode, 0 for
    an even one.

    Physics constants come from `floquet_lindblad.models.ats` (single source of truth).
    """
    import time

    import jax
    import jax.numpy as jnp
    import numpy as np

    import dynamiqs as dq
    from floquet_lindblad import KrylovSchurLindblad
    from floquet_lindblad.models.ats import (
        E_J,
        KAPPA_B,
        PHI_A,
        PHI_B,
        W_A,
        _derived_params,
        _two_mode_operators,
    )

    def py(v):
        """0-d jax/numpy scalar -> plain python float/complex."""
        a = np.asarray(v)
        return complex(a) if np.iscomplexobj(a) else float(a)

    lines = []

    def log(s):
        if verbose:
            print(s)
        lines.append(s)

    t0 = time.time()
    dims = (n_a, n_b)

    # --- plain RWA model: time independent ---------------------------------
    params, T_drive = _derived_params(
        alpha_sq, W_A, KAPPA_B, E_J, PHI_A, PHI_B, eps_p, n_periods=1
    )
    T_block = float(T_drive) if T_block is None else float(T_block)
    g, epsilon_d = params["g"], params["epsilon_d"]

    a, b = _two_mode_operators(n_a, n_b)
    H = epsilon_d / 2 * (b + dq.dag(b)) + g * (
        a @ a @ dq.dag(b) + dq.dag(a) @ dq.dag(a) @ b
    )
    
    Ka  = g**2 / W_A * (40*PHI_A**2 + 27*PHI_B**2) / (6*PHI_B**2)
    Kb  = g**2 / W_A * 2*PHI_B**2*(58*PHI_A**2 + 25*PHI_B**2) / (15*PHI_A**4)
    chi = g**2 / W_A * (135*PHI_A**2 + 92*PHI_B**2) / (15*PHI_A**2)
    Da  = -g**2 / W_A * (100*PHI_A**4 + 135*PHI_A**2*PHI_B**2 + 46*PHI_B**4) / (15*PHI_A**2*PHI_B**2)
    Db  = -g**2 / W_A * (135*PHI_A**4 + 324*PHI_A**2*PHI_B**2 + 100*PHI_B**4) / (30*PHI_A**4)

    na, nb = dq.dag(a) @ a, dq.dag(b) @ b
    H2 = Da*na + Db*nb - Ka/2 * dq.dag(a)@dq.dag(a)@a@a - Kb/2 * dq.dag(b)@dq.dag(b)@b@b - chi * na@nb
    H = H + H2

    jump_ops = [jnp.sqrt(params["kappa_1"]) * a, jnp.sqrt(params["kappa_b"]) * b]

    solver = KrylovSchurLindblad(
        H, jump_ops, T_block, dims=dims, rtol=rtol_loop, atol=atol_loop
    )
    x0 = solver.make_x0(seed=seed)
    t_build = time.time() - t0

    # --- phase 1: cycles at the loop tolerance -------------------------------
    same_tol = (rtol_loop, atol_loop) == (rtol_final, atol_final)
    if max_cycles_loop is None:
        max_cycles_loop = max(1, max_cycles // 2)
    t1 = time.time()
    out = solver.solve(
        x0,
        m=m,
        k=k,
        res_tol=res_tol if same_tol else res_tol_loop,
        max_cycles=max_cycles if same_tol else max_cycles_loop,
        log=log,
    )
    history = list(out["history"])
    n_cycles_loop = out["n_cycles"]
    t_loop = time.time() - t1

    # --- phase 2: polish at the final tolerance, resuming the kept subspace ---
    t2 = time.time()
    if not same_tol:
        log(f"-- switching integrator tolerance to rtol={rtol_final:g}, atol={atol_final:g}")
        solver.set_tolerance(rtol_final, atol_final)
        out = solver.solve(
            state=out["state"],
            m=m,
            k=k,
            res_tol=res_tol,
            max_cycles=max(1, max_cycles - n_cycles_loop),
            log=log,
        )
        history += list(out["history"])
    t_polish = time.time() - t2

    x_ritz = out["x_ritz"]
    gap = max(float(1 - complex(out["mu"]).real), 1e-14)
    log(f"-- final Rayleigh quotient on {n_blocks_final} block(s): "
        f"1 - mu = {gap:.2e} per block, 1 - mu**n ~ {n_blocks_final * gap:.2e} "
        f"({n_blocks_final * gap / rtol_final:.1e} x rtol_final)")

    # --- final numbers on the true propagator, tight tolerance ---------------
    t3 = time.time()
    res_1 = solver.residual_check(x_ritz, n_blocks=1, rtol=rtol_final, atol=atol_final)
    res_n = solver.residual_check(
        x_ritz, n_blocks=n_blocks_final, rtol=rtol_final, atol=atol_final
    )
    t_final = time.time() - t3

    # --- truncation check and parity of the target (Fock basis already) -----
    rho = np.asarray(solver.unvec(x_ritz))
    D = np.einsum("ij,ij->i", rho, rho.conj()).real.reshape(n_a, n_b)
    w_a = D.sum(axis=1) / D.sum()
    w_b = D.sum(axis=0) / D.sum()
    parity = np.kron((-1.0) ** np.arange(n_a), np.ones(n_b))
    rho_p = parity[:, None] * rho * parity[None, :]
    odd_fraction = float(np.linalg.norm(rho - rho_p) / (2 * np.linalg.norm(rho)))

    return {
        "model": "rwa",
        "alpha_sq": float(alpha_sq),
        "eps_p": float(eps_p),
        "n_a": int(n_a),
        "n_b": int(n_b),
        # eigenvalue from the Krylov iteration (one block) ...
        "mu": complex(out["mu"]),
        "rate_ritz": py(solver.rate_from_mu(out["mu"])),
        "res_rel": float(out["res_rel"]),
        "converged": bool(out["converged"]),
        # ... and from the final Rayleigh quotients at tight tolerance
        "mu_RQ_1": py(res_1["mu_RQ"]),
        "res_rel_1": py(res_1["res_rel"]),
        "mu_RQ_n": py(res_n["mu_RQ"]),
        "rate_bf": py(res_n["rate_RQ"]),  # named as in `solve_point_ks`
        "res_rel_n": py(res_n["res_rel"]),
        "n_blocks_final": int(n_blocks_final),
        "gap": gap,
        # cost
        "n_apply": int(out["n_apply"]),
        "n_cycles": len(history),
        "n_cycles_loop": int(n_cycles_loop),
        "res_hist": np.array([h["res_rel"] for h in history]),
        "apply_hist": np.array([h["n_apply"] for h in history]),
        "mu_hist": np.array([h["mu"] for h in history]),
        "log": lines,
        # truncation check, parity
        "fock_weight_a": w_a,
        "fock_weight_b": w_b,
        "tail_a": float(w_a[-k_top:].sum()),
        "tail_b": float(w_b[-k_top:].sum()),
        "k_top": int(k_top),
        "odd_fraction": odd_fraction,
        "params": {kk: py(v) for kk, v in params.items()},
        "T_block": T_block,
        "settings": {
            "n_a": n_a, "n_b": n_b, "m": m, "k": k, "T_block": T_block,
            "res_tol": res_tol, "res_tol_loop": res_tol_loop,
            "max_cycles": max_cycles, "max_cycles_loop": max_cycles_loop,
            "rtol_loop": rtol_loop, "atol_loop": atol_loop,
            "rtol_final": rtol_final, "atol_final": atol_final,
            "n_blocks_final": n_blocks_final, "seed": seed,
        },
        "timings_s": {
            "build": t_build,
            "loop": t_loop,
            "polish": t_polish,
            "final_RQ": t_final,
            "total": time.time() - t0,
        },
        "device": str(jax.devices()[0]),
    }


In [ ]:
def sweep_sequential(worker, points, common):
    """Solve `points` one after the other, on this job's single GPU.

    `worker` is passed in rather than referenced by name: it is defined in this
    notebook, so it ships by value with this function.
    """
    import time

    out = []
    for p in points:
        t0 = time.time()
        try:
            res = worker(**common, **p)
        except Exception as exc:  # noqa: BLE001 - one bad point must not sink the sweep
            res = {**p, "error": f"{type(exc).__name__}: {exc}"}
        res["wall_s"] = time.time() - t0
        out.append(res)
        print(f"alpha_sq = {p['alpha_sq']:5.2f} ({p['n_a']} x {p['n_b']})  "
              + (res["error"] if "error" in res
                 else f"rate_bf = {complex(res['rate_bf']).real:.6e}  "
                      f"res = {res['res_rel']:.1e}  apps = {res['n_apply']:d}"
                      + ("" if res["converged"] else "  (NOT converged)"))
              + f"  [{res['wall_s']:.0f} s]", flush=True)
    return out

### The sweep

In [ ]:
import pickle
import time

OUT = REPO / "results" / "sweep_alpha_RWA_0.pkl"
OUT.parent.mkdir(exist_ok=True)

results = {}
for name in ("small", "ats"):  # cheap one first
    t0 = time.time()
    results[name] = acr.run(
        sweep_sequential, run_krylov_schur_rwa, PLANS[name], COMMON,
        num_gpus=1,
        num_cpus=4,
        runtime_env=RUNTIME_ENV,
        poll_seconds=10,
    )
    print(f"\n[{name}] wall clock incl. job startup: {time.time() - t0:.1f} s "
          f"for {len(results[name])} point(s)\n")
    # Saved after each truncation, so a failure in the second keeps the first.
    with open(OUT, "wb") as f:
        pickle.dump({"results": results, "rate_one_mode": RATE_1M, "params": PARAMS,
                     "plans": PLANS, "common": COMMON, "n_a_ref": N_A_REF}, f)
    print(f"wrote {OUT}")

### Results

In [ ]:
import numpy as np

for name, rs in results.items():
    print(f"=== truncation: {name} ===")
    print(f"{'alpha_sq':>8} {'n_a x n_b':>10} {'rate_bf':>13} {'one mode':>11} {'ratio':>7} "
          f"{'res Krylov':>11} {'res RQ':>9} {'conv':>5} {'odd':>5} {'blocks':>7} "
          f"{'resolution':>11} {'GPU s':>7}")
    for r in rs:
        if "error" in r:
            print(f"{r['alpha_sq']:8.2f} {r['n_a']:4d} x{r['n_b']:4d}  FAILED: {r['error']}")
            continue
        rate = complex(r["rate_bf"]).real
        ref = RATE_1M[r["alpha_sq"]]
        resolution = (1 - complex(r["mu_RQ_n"]).real) / r["settings"]["rtol_final"]
        print(f"{r['alpha_sq']:8.2f} {r['n_a']:4d} x{r['n_b']:4d} {rate:13.5e} {ref:11.4e} "
              f"{rate / ref:7.4f} {r['res_rel']:11.2e} {abs(r['res_rel_n']):9.2e} "
              f"{('yes' if r['converged'] else 'NO'):>5} {r['odd_fraction']:5.2f} "
              f"{r['n_blocks_final']:7d} {resolution:11.1e} "
              f"{r['timings_s']['loop'] + r['timings_s']['polish'] + r['timings_s']['final_RQ']:7.1f}")
    print()

# Truncation effect: small vs ats, point by point.
if {"small", "ats"} <= results.keys():
    print(f"{'alpha_sq':>8} {'small / ats':>12}")
    for rs_, ra_ in zip(results["small"], results["ats"]):
        if "error" in rs_ or "error" in ra_:
            continue
        print(f"{rs_['alpha_sq']:8.2f} "
              f"{complex(rs_['rate_bf']).real / complex(ra_['rate_bf']).real:12.5f}")

### Bit-flip rate vs cat size

In [ ]:
import matplotlib.pyplot as plt

# Categorical hues in fixed order (see the dataviz palette): blue, orange. The
# one-mode reference is a neutral ink line, not a series hue.
COLORS = {"ats": "#2a78d6", "small": "#eb6834"}
LABELS = {"ats": "RWA, ATS truncation", "small": "RWA, small truncation"}
MARKERS = {"ats": "o", "small": "s"}
SIZES = {"ats": 6, "small": 9}  # small drawn first and larger, so both stay visible

a_ref = np.array(ALPHA_SQ_LIST)
rate_ref = np.array([RATE_1M[a] for a in ALPHA_SQ_LIST])

fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.0), constrained_layout=True)
axes[0].plot(a_ref, rate_ref/2, color="0.25", lw=1.2, label=f"one mode (eigs, n_a = {N_A_REF})")

for name, rs in results.items():
    ok = [r for r in rs if "error" not in r]
    a = np.array([r["alpha_sq"] for r in ok])
    rate = np.array([complex(r["rate_bf"]).real for r in ok])/2
    conv = np.array([r["converged"] for r in ok])
    ratio = rate / (np.array([RATE_1M[x] for x in a])/2)
    c, mk, ms = COLORS[name], MARKERS[name], SIZES[name]
    for ax, y in ((axes[0], rate), (axes[1], ratio)):
        ax.plot(a[conv], y[conv], mk, ms=ms, color=c, mec="white", mew=1.5,
                label=LABELS[name], zorder=3)
        if (~conv).any():
            ax.plot(a[~conv], y[~conv], mk, ms=ms, mfc="none", mec=c, mew=1.5,
                    label=f"{LABELS[name]}, not converged", zorder=3)
    if name == 'ats':
        k1 = np.array([r["params"]["kappa_1"] for r in ok])
        k2 = np.array([r["params"]["kappa_2"] for r in ok])
        pert_th = k1 * a * np.exp(-4 * a) + k1**2 / (2 * k2) * np.exp(-2 * a)
        axes[0].plot(a, pert_th, ls=":", color="k", label="Perturbative (Dubovitskii)")

axes[0].set_yscale("log")
axes[0].set_xlabel(r"$|\alpha|^2$")
axes[0].set_ylabel("bit-flip rate")
axes[0].set_title("bit-flip rate", fontsize=9)
axes[0].legend(fontsize=8, frameon=False)

axes[1].axhline(1, color="0.6", lw=0.8, ls="--")
axes[1].set_xlabel(r"$|\alpha|^2$")
axes[1].set_ylabel("rate / one-mode rate")
axes[1].set_title("RWA relative to the one-mode model", fontsize=9)
axes[1].legend(fontsize=8, frameon=False)

for ax in axes:
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(True, which="major", lw=0.4, color="0.9")
    ax.set_axisbelow(True)
plt.show()

In [ ]:
import json


def to_jsonable(obj):
    """numpy-only copy of floquet_lindblad.io.to_jsonable (that module imports jax,
    which this client does not have). Complex -> {"real", "imag"}, arrays -> lists."""
    if isinstance(obj, dict):
        return {k: to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return to_jsonable(obj.tolist())
    if isinstance(obj, (np.complexfloating, complex)):
        return {"real": float(obj.real), "imag": float(obj.imag)}
    if isinstance(obj, (np.floating, float)):
        return float(obj)
    if isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    if isinstance(obj, (np.integer, int)):
        return int(obj)
    return obj


out_path = REPO / "results" / "sweep_alpha_eps_p_RWA.json"
out_path.parent.mkdir(exist_ok=True)
with open(out_path, "w") as f:
    json.dump(to_jsonable(results["ats"]), f, indent=2)
print(f"wrote {len(results['ats'])} points to {out_path}")


In [ ]:
import matplotlib.pyplot as plt

# Categorical hues in fixed order (see the dataviz palette): blue, orange. The
# one-mode reference is a neutral ink line, not a series hue.
COLORS = {"ats": "#2a78d6", "small": "#eb6834"}
LABELS = {"ats": "RWA, ATS truncation", "small": "RWA, small truncation"}
MARKERS = {"ats": "o", "small": "s"}
SIZES = {"ats": 6, "small": 9}  # small drawn first and larger, so both stay visible

a_ref = np.array(ALPHA_SQ_LIST)
rate_ref = np.array([RATE_1M[a] for a in ALPHA_SQ_LIST])

fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.0), constrained_layout=True)
axes[0].plot(a_ref, rate_ref/2, color="0.25", lw=1.2, label=f"one mode (eigs, n_a = {N_A_REF})")

for name, rs in results.items():
    ok = [r for r in rs if "error" not in r]
    a = np.array([r["alpha_sq"] for r in ok])
    rate = np.array([complex(r["rate_bf"]).real for r in ok])/2
    conv = np.array([r["converged"] for r in ok])
    ratio = rate / (np.array([RATE_1M[x] for x in a])/2)
    c, mk, ms = COLORS[name], MARKERS[name], SIZES[name]
    for ax, y in ((axes[0], rate), (axes[1], ratio)):
        ax.plot(a[conv], y[conv], mk, ms=ms, color=c, mec="white", mew=1.5,
                label=LABELS[name], zorder=3)
        if (~conv).any():
            ax.plot(a[~conv], y[~conv], mk, ms=ms, mfc="none", mec=c, mew=1.5,
                    label=f"{LABELS[name]}, not converged", zorder=3)
    if name == 'ats':
        k1 = np.array([r["params"]["kappa_1"] for r in ok])
        k2 = np.array([r["params"]["kappa_2"] for r in ok])
        pert_th = k1 * a * np.exp(-4 * a) + k1**2 / (2 * k2) * np.exp(-2 * a)
        axes[0].plot(a, pert_th, ls=":", color="k", label="Perturbative (Dubovitskii)")

axes[0].set_yscale("log")
axes[0].set_xlabel(r"$|\alpha|^2$")
axes[0].set_ylabel("bit-flip rate")
axes[0].set_title("bit-flip rate", fontsize=9)
axes[0].legend(fontsize=8, frameon=False)

axes[1].axhline(1, color="0.6", lw=0.8, ls="--")
axes[1].set_xlabel(r"$|\alpha|^2$")
axes[1].set_ylabel("rate / one-mode rate")
axes[1].set_title("RWA relative to the one-mode model", fontsize=9)
axes[1].legend(fontsize=8, frameon=False)

for ax in axes:
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(True, which="major", lw=0.4, color="0.9")
    ax.set_axisbelow(True)
plt.show()

### Truncation check

As in `sweep_alpha.ipynb`, the weight in the top `k_top` levels of each mode should sit far below the residual we are quoting. `odd` is the parity-odd fraction of the target, which should be 1 for a bit-flip mode.

In [ ]:
TOL = 1e-11  # same absolute criterion as the ATS sweep

for name, rs in results.items():
    ok = [r for r in rs if "error" not in r]
    print(f"=== truncation: {name} ===")
    print(f"{'alpha_sq':>9} {'n_a':>4} {'n_b':>4} {'tail_a':>10} {'tail_b':>10} "
          f"{'odd':>5} {'verdict':>8}")
    worst = []
    for r in ok:
        passed = max(r["tail_a"], r["tail_b"]) < TOL
        if not passed:
            worst.append(r)
        print(f"{r['alpha_sq']:9.2f} {r['n_a']:4d} {r['n_b']:4d} "
              f"{r['tail_a']:10.2e} {r['tail_b']:10.2e} {r['odd_fraction']:5.2f} "
              f"{('PASS' if passed else 'FAIL'):>8}")
    print(f"top-{ok[0]['k_top']} weight against tol = {TOL:.0e}: "
          + ("all points PASS" if not worst else
             "raise n_a / n_b for alpha_sq = "
             + ", ".join(f"{r['alpha_sq']:.2f}" for r in worst)) + "\n")